# Player stat forecasts in the 2026 Six Nations

1. Which players are forecast to produce more of a stat than their positional peers?
2. For which stats does using player history improve forecast accuracy over a positional average?

We fit on all competitions before the opening match and hold the model fixed through the tournament. Each test observation is one recorded player appearance.

In [1]:
from pathlib import Path

import pandas as pd

from six_nations_fantasy.data import load_dataset
from six_nations_fantasy.data.lineup import substitute_status
from six_nations_fantasy.models import PositionShrinkageMean
from six_nations_fantasy.models.position_shrinkage import STARTING_POSITIONS

data_dir = Path("../data/rebuild")  # relative to this notebook
data = load_dataset(data_dir)
players = pd.read_parquet(data_dir / "players.parquet")

In [2]:
opening_day = "2026-02-05"
day_after_final = "2026-03-15"
matches = data.matches
training = data.select_matches(matches.loc[matches["date"] < opening_day, "match_id"])

tournament = matches["competition"].eq("Six Nations") & matches["date"].between(
    opening_day, day_after_final, inclusive="left"
)
test = data.select_matches(matches.loc[tournament, "match_id"]).player_matches()
fixtures = test[["match_id", "player_id", "number"]].assign(is_substitute=substitute_status(test))

## Forecast each stat

For each appearance, save the player forecast and the training average for the same starting position. All substitutes share a single bench baseline. The model uses the same shrinkage setting ($k=20$) for every stat. We keep the observed outcome alongside the forecasts for evaluation.

In [3]:
stats = [
    "conversions", "penalty_goals", "drop_goals", "carries", "clean_breaks",
    "completed_tackles", "turnovers_conceded", "turnovers_won", "total_kicks",
    "passes", "carried_meters", "offloads", "defenders_beaten", "try_assists",
    "tries", "carries_per_minute", "total_tackles", "dominant_tackles",
    "ruck_turnovers", "lineouts_won", "tackles_per_minute",
]

position = pd.to_numeric(fixtures["number"]).map(STARTING_POSITIONS)
baseline_group = position.mask(fixtures["is_substitute"].fillna(False), "Substitute")

def forecast_stat(stat):
    model = PositionShrinkageMean(k=20, scorer=lambda rows: rows[stat])
    model.fit(training)
    return fixtures[["match_id", "player_id"]].assign(
        stat=stat,
        baseline_group=baseline_group,
        forecast=model.predict(fixtures),
        baseline=model.baselines(fixtures),
        actual=test[stat],
    )

forecasts = pd.concat([forecast_stat(stat) for stat in stats], ignore_index=True)

## Which players have high predicted volume relative to their peers?

In the following, we show the forecast-to-baseline ratio for a selected stat. A large ratio can correspond to a small difference in events per match, especially for rare stats, so this is not necessarily a great selection criterion (for fantasy, perhaps expected extra points per match would be more relevant). 

In [7]:
stat_to_inspect = "carries"
leaderboard = (
    forecasts.loc[forecasts["stat"].eq(stat_to_inspect)]
    .groupby(["player_id", "baseline_group"])
    .agg(appearances=("match_id", "size"), forecast=("forecast", "mean"), baseline=("baseline", "mean"))
    .join(players["name"], on="player_id")
)
baseline = leaderboard["baseline"]
leaderboard["forecast_ratio"] = leaderboard["forecast"] / baseline.where(baseline > 0)
leaderboard["extra_per_match"] = leaderboard["forecast"] - baseline
leaderboard.nlargest(10, "forecast_ratio")

,,appearances,forecast,baseline,name,forecast_ratio,extra_per_match
player_id,baseline_group,,,,,,
137493,Scrum-half,5,10.990536,5.112144,Antoine Dupont,2.149888,5.878392
196669,Prop,4,7.813404,3.978255,Zander Fagerson,1.964028,3.835149
196773,Prop,3,7.708638,3.978255,Pierre Schoeman,1.937693,3.730383
196669,Substitute,1,4.189656,2.322415,Zander Fagerson,1.804008,1.867241
165007,Prop,5,7.007849,3.978255,Ellis Genge,1.761538,3.029594
196773,Substitute,2,4.014943,2.322415,Pierre Schoeman,1.728779,1.692528
177559,Scrum-half,3,8.665577,5.112144,Alex Mitchell,1.695096,3.553433
134112,Substitute,3,3.748769,2.322415,Emmanuel Meafou,1.614168,1.426354
190344,Fly-half,4,10.883182,6.812637,Matthieu Jalibert,1.597499,4.070545


## For which stats does player history improve the forecast?

The **mean squared error skill score (MSESS)** measures the proportional reduction in mean squared error relative to the positional baseline. Compute the two errors across the same held-out appearances, then take their ratio.

$$\mathrm{MSESS} = 1 - \frac{\sum_i (y_i - \hat y_i)^2}{\sum_i (y_i - b_i)^2}$$

Here $y_i$ is the observed stat, $\hat y_i$ the player forecast and $b_i$ the baseline. A score of 0.10 means a 10% reduction in squared error. Zero means equal error and a negative value favors the baseline. A perfect baseline makes the score undefined. The RMSE columns give the typical scale of the errors in the original stat units.

In [ ]:
baseline_groups = [
    "Prop", "Hooker", "Second-row",
    "Back-row", "Scrum-half", "Fly-half",
    "Back Three", "Centre", "Substitute"
]

paired = forecasts.dropna(subset=["actual", "forecast", "baseline"]).copy()
paired["forecast_sq_error"] = (paired["actual"] - paired["forecast"]) ** 2
paired["baseline_sq_error"] = (paired["actual"] - paired["baseline"]) ** 2

accuracy = paired.groupby(["stat", "baseline_group"]).agg(
    appearances=("match_id", "size"),
    forecast_mse=("forecast_sq_error", "mean"),
    baseline_mse=("baseline_sq_error", "mean"),
).reindex(pd.MultiIndex.from_product([stats, baseline_groups]))
baseline_mse = accuracy["baseline_mse"]
accuracy["msess"] = 1 - accuracy["forecast_mse"] / baseline_mse.where(baseline_mse > 0)
accuracy["forecast_rmse"] = accuracy["forecast_mse"] ** 0.5
accuracy["baseline_rmse"] = baseline_mse ** 0.5
accuracy[["appearances", "forecast_rmse", "baseline_rmse", "msess"]].sort_values(
    "msess", ascending=False
).nlargest(15, columns="msess")

,,appearances,forecast_rmse,baseline_rmse,msess
lineouts_won,Second-row,60,0.000253,0.000821,0.904767
drop_goals,Centre,60,0.000127,0.000410,0.904167
lineouts_won,Prop,60,0.001106,0.003146,0.876432
penalty_goals,Scrum-half,30,0.047300,0.097648,0.765358
conversions,Back Three,90,0.553868,1.116854,0.754065
carries,Scrum-half,30,2.626488,4.162600,0.601873
carries_per_minute,Scrum-half,30,0.044521,0.065410,0.536731
carries,Prop,60,2.532488,3.634443,0.514466
carries_per_minute,Prop,60,0.042165,0.059222,0.493076
lineouts_won,Substitute,240,1.278845,1.746547,0.463864


Read the first table when comparing players for a particular stat. Read the second when deciding whether individual player history helps forecast that stat across this tournament. For fantasy selection, a further question is whether the players ranked highest before kickoff actually outperformed their positional peers.

In [9]:
paired

,match_id,player_id,stat,baseline_group,forecast,baseline,actual,forecast_sq_error,baseline_sq_error
0,290465,104627,conversions,Prop,0.000000,0.000000,0.000000,0.000000,0.000000
1,290465,150335,conversions,Hooker,0.000000,0.000000,0.000000,0.000000,0.000000
2,290465,158534,conversions,Prop,0.000000,0.000000,0.000000,0.000000,0.000000
3,290465,198942,conversions,Second-row,0.000000,0.000000,0.000000,0.000000,0.000000
4,290465,177800,conversions,Second-row,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...
14485,290479,240631,tackles_per_minute,Substitute,0.167405,0.135357,0.400000,0.054100,0.070036
14486,290479,104322,tackles_per_minute,Substitute,0.156070,0.135357,0.222222,0.004376,0.007546
14487,290479,239634,tackles_per_minute,Substitute,0.160338,0.135357,0.142857,0.000306,0.000056
14488,290479,113737,tackles_per_minute,Substitute,0.114319,0.135357,0.000000,0.013069,0.018321
